# 🎭 Generative Adversarial Networks (GANs)

This notebook builds deep intuition for:
1. **The adversarial game** — why two networks fighting produces creativity
2. **Generator & Discriminator** — the counterfeiter and the detective
3. **The minimax objective** — what GANs actually optimize
4. **Training dynamics** — why GANs are notoriously hard to train
5. **Mode collapse** — when the generator gets lazy
6. **Wasserstein distance** — a better way to measure distribution similarity
7. **From theory to practice** — building a GAN from scratch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import norm, wasserstein_distance

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

np.random.seed(42)

## 1. The Counterfeiter and the Detective

Imagine a **counterfeiter** trying to produce fake banknotes. At first, the fakes are terrible —
wrong color, wrong texture, wrong watermark. But there's also a **detective** whose job is to
tell real banknotes from fake ones.

Every time the detective catches a fake, the counterfeiter learns what went wrong and improves.
Every time the counterfeiter fools the detective, the detective sharpens their skills.

Over time, this adversarial pressure pushes the counterfeiter to produce **perfect** fakes —
and the detective can do no better than random guessing.

This is exactly how a GAN works:
- **Generator** $G$ = counterfeiter (turns random noise into fake data)
- **Discriminator** $D$ = detective (classifies data as real or fake)

Let's watch this unfold with simple 1D distributions.

In [ ]:
# The adversarial game: watching a generator learn

# Real data distribution: mixture of two Gaussians
def p_data(x):
    return 0.5 * norm.pdf(x, -2, 0.6) + 0.5 * norm.pdf(x, 2, 0.6)

def sample_real(n):
    """Sample from the real distribution."""
    components = np.random.choice([0, 1], size=n)
    return np.where(components == 0, 
                    np.random.normal(-2, 0.6, n),
                    np.random.normal(2, 0.6, n))

# Simulate the generator as a Gaussian that we'll optimize
# (In real GANs, this is a neural network)
x_plot = np.linspace(-6, 6, 500)

# Stages of the generator learning
generator_stages = [
    {'mu': 0, 'sigma': 3.0, 'label': 'Step 0: Random guess'},
    {'mu': 0, 'sigma': 2.0, 'label': 'Step 50: Narrowing down'},
    {'mu': 1.5, 'sigma': 1.2, 'label': 'Step 150: Found one mode'},
    {'mu': 2, 'sigma': 0.6, 'label': 'Step 300: Near one mode'},
    {'mu': -0.5, 'sigma': 2.5, 'label': 'Step 500: Trying both modes'},
]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, stage in enumerate(generator_stages):
    ax = axes[i]
    p_real = p_data(x_plot)
    p_gen = norm.pdf(x_plot, stage['mu'], stage['sigma'])
    
    ax.fill_between(x_plot, p_real, alpha=0.3, color='steelblue', label='Real data')
    ax.plot(x_plot, p_gen, 'r-', linewidth=2, label='Generator')
    ax.set_title(stage['label'], fontsize=10)
    ax.set_ylim(0, 0.45)
    if i == 0:
        ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('The Generator\'s Journey: Learning to Match the Real Distribution', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("The generator starts with random noise and gradually learns to mimic the real data.")
print("Notice: a single Gaussian CANNOT capture both modes — this is a preview of mode collapse!")

## 2. The Mathematics of Adversarial Training

The GAN objective is a **minimax game**:

$$\min_G \max_D \underbrace{\mathbb{E}_{x \sim p_{data}}[\log D(x)]}_{\text{D's reward for catching real}} + \underbrace{\mathbb{E}_{z \sim p_z}[\log(1 - D(G(z)))]}_{\text{D's reward for catching fakes}}$$

**What each player wants:**
- **Discriminator** maximizes: push $D(x_{real}) \to 1$ and $D(x_{fake}) \to 0$
- **Generator** minimizes: push $D(x_{fake}) \to 1$ (fool the discriminator)

### The Optimal Discriminator

For a fixed generator, the **optimal discriminator** has a beautiful closed form:

$$D^*(x) = \frac{p_{data}(x)}{p_{data}(x) + p_g(x)}$$

This is simply the Bayes-optimal classifier! It outputs the probability that $x$ came from the real data.

In [ ]:
# Visualizing the optimal discriminator

def optimal_discriminator(x, p_real_fn, p_gen_fn):
    """D*(x) = p_data(x) / (p_data(x) + p_g(x))"""
    p_r = p_real_fn(x)
    p_g = p_gen_fn(x)
    return p_r / (p_r + p_g + 1e-10)

x = np.linspace(-6, 6, 500)

# Case 1: Generator far from data
p_gen_bad = lambda x: norm.pdf(x, 4, 0.5)
# Case 2: Generator partially overlapping
p_gen_ok = lambda x: norm.pdf(x, 1, 1.5)
# Case 3: Generator = data (perfect)
p_gen_perfect = lambda x: p_data(x)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

cases = [
    (p_gen_bad, 'Generator far from data'),
    (p_gen_ok, 'Generator partially matches'),
    (p_gen_perfect, 'Generator = data (PERFECT)'),
]

for i, (p_gen, title) in enumerate(cases):
    # Top row: distributions
    axes[0, i].fill_between(x, p_data(x), alpha=0.3, color='steelblue', label='p_data')
    axes[0, i].plot(x, p_gen(x), 'r-', linewidth=2, label='p_g')
    axes[0, i].set_title(title)
    axes[0, i].legend(fontsize=9)
    axes[0, i].grid(True, alpha=0.3)
    
    # Bottom row: optimal discriminator
    d_star = optimal_discriminator(x, p_data, p_gen)
    axes[1, i].plot(x, d_star, 'g-', linewidth=2)
    axes[1, i].axhline(y=0.5, color='gray', linestyle=':', alpha=0.7, label='D=0.5')
    axes[1, i].set_ylim(-0.05, 1.05)
    axes[1, i].set_title(f'Optimal D*(x)')
    axes[1, i].set_ylabel('D*(x)')
    axes[1, i].legend(fontsize=9)
    axes[1, i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("🔑 Key Insight:")
print("When the generator perfectly matches the data, D*(x) = 0.5 EVERYWHERE.")
print("The detective literally cannot tell real from fake — this is the Nash equilibrium!")

## 3. What Does the Generator Actually Minimize?

Here's the deep connection to information theory. When the discriminator is optimal, the generator minimizes:

$$V(D^*, G) = 2 \cdot JSD(p_{data} \| p_g) - \log 4$$

where **JSD** is the **Jensen-Shannon Divergence**:

$$JSD(P \| Q) = \frac{1}{2} D_{KL}\left(P \| \frac{P+Q}{2}\right) + \frac{1}{2} D_{KL}\left(Q \| \frac{P+Q}{2}\right)$$

The JSD is a **symmetrized, bounded** version of KL divergence:
- $0 \leq JSD \leq \log 2$
- $JSD = 0$ iff $P = Q$ (distributions are identical)
- Unlike KL, it's symmetric: $JSD(P\|Q) = JSD(Q\|P)$

In [ ]:
# Computing JSD as the generator improves

def compute_jsd(p, q, dx):
    """Compute Jensen-Shannon divergence between discrete distributions."""
    m = 0.5 * (p + q)
    # Avoid log(0)
    kl_pm = np.sum(np.where(p > 1e-10, p * np.log(p / (m + 1e-10)), 0)) * dx
    kl_qm = np.sum(np.where(q > 1e-10, q * np.log(q / (m + 1e-10)), 0)) * dx
    return 0.5 * kl_pm + 0.5 * kl_qm

# Simulate generator getting better over time
# Generator starts far away and gradually moves toward the data
x = np.linspace(-8, 8, 1000)
dx = x[1] - x[0]
p_real = p_data(x)

# Generator evolution: starts as N(5, 2), gradually becomes p_data
n_steps = 100
jsds = []
value_fns = []

for step in range(n_steps):
    t = step / (n_steps - 1)  # 0 to 1
    # Interpolate from bad generator to perfect generator
    p_gen = (1 - t) * norm.pdf(x, 5, 2) + t * p_data(x)
    
    jsd = compute_jsd(p_real, p_gen, dx)
    jsds.append(jsd)
    value_fns.append(2 * jsd - np.log(4))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(jsds, 'b-', linewidth=2)
ax1.set_xlabel('Training Step (simulated)')
ax1.set_ylabel('JSD(p_data || p_g)')
ax1.set_title('Jensen-Shannon Divergence During Training')
ax1.axhline(y=0, color='green', linestyle='--', alpha=0.7, label='JSD=0 (perfect)')
ax1.axhline(y=np.log(2), color='red', linestyle='--', alpha=0.7, label=f'JSD=log(2)≈{np.log(2):.3f} (worst)')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(value_fns, 'r-', linewidth=2)
ax2.set_xlabel('Training Step (simulated)')
ax2.set_ylabel('V(D*, G)')
ax2.set_title('Value Function = 2·JSD - log(4)')
ax2.axhline(y=-np.log(4), color='green', linestyle='--', alpha=0.7, label=f'V*=-log(4)≈{-np.log(4):.3f}')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"JSD ranges from 0 (identical) to log(2)≈{np.log(2):.4f} (maximally different)")
print(f"Value function minimum: -log(4)≈{-np.log(4):.4f} (achieved when p_g = p_data)")
print(f"\n🔑 The GAN is minimizing the JSD between real and generated distributions!")

## 4. Training a Simple GAN

Let's implement a GAN from scratch with numpy. We'll use:
- **Real data:** Mixture of 2 Gaussians
- **Generator:** Small MLP (noise → hidden → output)
- **Discriminator:** Small MLP (input → hidden → probability)

We'll train with the standard alternating optimization:
1. Train D to classify real vs fake
2. Train G to fool D
3. Repeat

In [ ]:
# Simple 1D GAN from scratch

np.random.seed(42)

# --- Neural network helpers ---
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def relu(x):
    return np.maximum(0, x)

def relu_grad(x):
    return (x > 0).astype(float)

# --- Real data: mixture of Gaussians ---
def sample_real_data(n):
    idx = np.random.choice([0, 1], size=n)
    return np.where(idx == 0, 
                    np.random.normal(-2, 0.5, n),
                    np.random.normal(2, 0.5, n)).reshape(-1, 1)

# --- Network architecture ---
noise_dim = 4
hidden_dim = 16

# Generator weights: noise_dim -> hidden_dim -> 1
Wg1 = np.random.randn(noise_dim, hidden_dim) * 0.1
bg1 = np.zeros((1, hidden_dim))
Wg2 = np.random.randn(hidden_dim, 1) * 0.1
bg2 = np.zeros((1, 1))

# Discriminator weights: 1 -> hidden_dim -> 1
Wd1 = np.random.randn(1, hidden_dim) * 0.1
bd1 = np.zeros((1, hidden_dim))
Wd2 = np.random.randn(hidden_dim, 1) * 0.1
bd2 = np.zeros((1, 1))

# --- Forward passes ---
def generator_forward(z):
    h = relu(z @ Wg1 + bg1)
    return h @ Wg2 + bg2

def discriminator_forward(x):
    h = relu(x @ Wd1 + bd1)
    return sigmoid(h @ Wd2 + bd2)

# --- Training ---
lr_d = 0.01
lr_g = 0.01
batch_size = 256
n_steps = 3000

d_losses = []
g_losses = []
snapshots = {}

for step in range(n_steps):
    # === Train Discriminator ===
    real = sample_real_data(batch_size)
    noise = np.random.randn(batch_size, noise_dim)
    
    # Generator forward (no grad for G)
    g_h = relu(noise @ Wg1 + bg1)
    fake = g_h @ Wg2 + bg2
    
    # D forward on real
    d_h_real = relu(real @ Wd1 + bd1)
    d_out_real = sigmoid(d_h_real @ Wd2 + bd2)
    
    # D forward on fake
    d_h_fake = relu(fake @ Wd1 + bd1)
    d_out_fake = sigmoid(d_h_fake @ Wd2 + bd2)
    
    # D loss: -[log(D(real)) + log(1-D(fake))]
    d_loss = -np.mean(np.log(d_out_real + 1e-8) + np.log(1 - d_out_fake + 1e-8))
    d_losses.append(d_loss)
    
    # D backward
    # d(loss)/d(d_out_real) = -1/(d_out_real)
    dd_out_real = -1.0 / (d_out_real + 1e-8) / batch_size
    dd_pre_real = dd_out_real * d_out_real * (1 - d_out_real)  # sigmoid grad
    dWd2_real = d_h_real.T @ dd_pre_real
    dbd2_real = dd_pre_real.sum(axis=0, keepdims=True)
    dd_h_real = (dd_pre_real @ Wd2.T) * relu_grad(real @ Wd1 + bd1)
    dWd1_real = real.T @ dd_h_real
    dbd1_real = dd_h_real.sum(axis=0, keepdims=True)
    
    # d(loss)/d(d_out_fake) = 1/(1-d_out_fake)
    dd_out_fake = 1.0 / (1 - d_out_fake + 1e-8) / batch_size
    dd_pre_fake = dd_out_fake * d_out_fake * (1 - d_out_fake)
    dWd2_fake = d_h_fake.T @ dd_pre_fake
    dbd2_fake = dd_pre_fake.sum(axis=0, keepdims=True)
    dd_h_fake = (dd_pre_fake @ Wd2.T) * relu_grad(fake @ Wd1 + bd1)
    dWd1_fake = fake.T @ dd_h_fake
    dbd1_fake = dd_h_fake.sum(axis=0, keepdims=True)
    
    Wd2 -= lr_d * (dWd2_real + dWd2_fake)
    bd2 -= lr_d * (dbd2_real + dbd2_fake)
    Wd1 -= lr_d * (dWd1_real + dWd1_fake)
    bd1 -= lr_d * (dbd1_real + dbd1_fake)
    
    # === Train Generator (non-saturating loss: maximize log D(G(z))) ===
    noise = np.random.randn(batch_size, noise_dim)
    g_h = relu(noise @ Wg1 + bg1)
    fake = g_h @ Wg2 + bg2
    
    d_h = relu(fake @ Wd1 + bd1)
    d_out = sigmoid(d_h @ Wd2 + bd2)
    
    g_loss = -np.mean(np.log(d_out + 1e-8))
    g_losses.append(g_loss)
    
    # G backward (through D, but only update G)
    dg_out = -1.0 / (d_out + 1e-8) / batch_size
    dg_pre = dg_out * d_out * (1 - d_out)
    dg_h = (dg_pre @ Wd2.T) * relu_grad(fake @ Wd1 + bd1)
    dfake = dg_h @ Wd1.T  # gradient w.r.t. fake samples
    
    dWg2 = g_h.T @ dfake
    dbg2 = dfake.sum(axis=0, keepdims=True)
    dg_hidden = (dfake @ Wg2.T) * relu_grad(noise @ Wg1 + bg1)
    dWg1 = noise.T @ dg_hidden
    dbg1 = dg_hidden.sum(axis=0, keepdims=True)
    
    Wg2 -= lr_g * dWg2
    bg2 -= lr_g * dbg2
    Wg1 -= lr_g * dWg1
    bg1 -= lr_g * dbg1
    
    if step in [0, 500, 1000, 2000, 2999]:
        vis_noise = np.random.randn(2000, noise_dim)
        vis_fake = generator_forward(vis_noise)
        snapshots[step] = vis_fake.flatten()

print("Training complete!")
print(f"Final D loss: {d_losses[-1]:.4f}, G loss: {g_losses[-1]:.4f}")

In [ ]:
# Visualize training progress

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
x_plot = np.linspace(-6, 6, 200)

for i, (step, samples) in enumerate(snapshots.items()):
    row, col = i // 3, i % 3
    ax = axes[row, col]
    ax.hist(sample_real_data(2000).flatten(), bins=50, density=True, 
            alpha=0.4, color='steelblue', label='Real data')
    ax.hist(samples, bins=50, density=True, 
            alpha=0.4, color='coral', label='Generated')
    ax.set_title(f'Step {step}', fontsize=12)
    ax.set_xlim(-6, 6)
    if i == 0:
        ax.legend()
    ax.grid(True, alpha=0.3)

# Loss curves in last subplot
ax = axes[1, 2]
ax.plot(d_losses[::10], alpha=0.7, label='D loss', color='steelblue')
ax.plot(g_losses[::10], alpha=0.7, label='G loss', color='coral')
ax.set_title('Training Losses')
ax.set_xlabel('Step (×10)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('GAN Training: Generated vs Real Distribution Over Time', fontsize=14)
plt.tight_layout()
plt.show()

print("\n🔑 Key Observations:")
print("1. The generator starts with random noise and gradually shapes into the target")
print("2. Losses oscillate — this is NORMAL for GAN training (it's a game, not optimization)")
print("3. The generator may not capture both modes perfectly — this hints at mode collapse")

## 5. Mode Collapse — When the Generator Gets Lazy

**Mode collapse** is the most infamous GAN failure mode. Instead of learning the full data distribution,
the generator discovers it can get a low loss by producing samples from just **one mode** of the data.

**Why does this happen?**
- The generator minimizes something close to **reverse KL**: $D_{KL}(p_g \| p_{data})$
- Reverse KL is **mode-seeking** — it's not penalized for missing modes, only for placing mass where $p_{data}$ is low
- Generating between modes is risky (low $p_{data}$, discriminator catches you easily)
- Concentrating on one mode is safe (high $p_{data}$, hard for discriminator to reject)

Let's demonstrate this dramatically.

In [ ]:
# Mode collapse demonstration in 2D

np.random.seed(123)

# Real data: 8 Gaussian clusters arranged in a circle
def sample_ring_data(n, n_modes=8, radius=3, std=0.2):
    """Sample from n_modes Gaussians arranged in a circle."""
    angles = np.linspace(0, 2*np.pi, n_modes, endpoint=False)
    centers = np.column_stack([radius * np.cos(angles), radius * np.sin(angles)])
    
    idx = np.random.randint(0, n_modes, n)
    samples = centers[idx] + np.random.randn(n, 2) * std
    return samples, centers

# Simulate mode collapse: generator visits modes one at a time
real_data, centers = sample_ring_data(1000)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Simulated mode collapse stages
collapse_stages = [
    ('Early training', np.random.randn(500, 2) * 1.5),
    ('Partial collapse\n(3/8 modes)', np.vstack([
        centers[0] + np.random.randn(200, 2) * 0.2,
        centers[3] + np.random.randn(200, 2) * 0.2,
        centers[6] + np.random.randn(100, 2) * 0.2
    ])),
    ('Severe collapse\n(1/8 modes)', centers[2] + np.random.randn(500, 2) * 0.2),
    ('Ideal result\n(all 8 modes)', sample_ring_data(500)[0]),
]

for i, (title, gen_data) in enumerate(collapse_stages):
    ax = axes[i]
    ax.scatter(real_data[:, 0], real_data[:, 1], alpha=0.2, s=5, c='steelblue', label='Real')
    ax.scatter(gen_data[:, 0], gen_data[:, 1], alpha=0.5, s=15, c='coral', label='Generated')
    
    # Mark true centers
    ax.scatter(centers[:, 0], centers[:, 1], c='black', s=50, marker='x', linewidths=2)
    
    ax.set_title(title, fontsize=11)
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.set_aspect('equal')
    if i == 0:
        ax.legend(fontsize=8, loc='upper left')
    ax.grid(True, alpha=0.3)

plt.suptitle('Mode Collapse: Generator Fails to Cover All Modes', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("🔑 Mode Collapse Explained:")
print("• The generator finds it 'safer' to produce samples from a few modes")
print("• It avoids the space between modes (where D can easily reject samples)")
print("• The D-G game oscillates: G jumps between modes without covering all of them")
print("• This is a fundamental limitation of the minimax training paradigm")

## 6. The Wasserstein Distance — A Better Metric

The root cause of many GAN problems is that **JSD is a poor training signal** when distributions
don't overlap much. Let's see why, and how the **Wasserstein distance** (Earth Mover's Distance) fixes this.

### Intuition: Moving Dirt

Think of two distributions as piles of dirt. The Wasserstein distance asks:
> **What is the minimum total work (mass × distance) to reshape one pile into the other?**

$$W_1(P, Q) = \inf_{\gamma \in \Pi(P,Q)} \mathbb{E}_{(x,y) \sim \gamma}[\|x - y\|]$$

### Why is this better than JSD?

| Scenario | JSD | Wasserstein |
|----------|-----|-------------|
| Distributions overlap | Smooth gradient | Smooth gradient |
| Distributions don't overlap | **Constant log(2), zero gradient!** | **Still provides gradient** |
| Distributions getting closer | May not change | Decreases smoothly |

In [ ]:
# JSD vs Wasserstein: the critical difference

def compute_jsd_gaussians(mu1, s1, mu2, s2, n_points=5000):
    """Compute JSD between two Gaussians numerically."""
    lo = min(mu1, mu2) - 5*max(s1, s2)
    hi = max(mu1, mu2) + 5*max(s1, s2)
    x = np.linspace(lo, hi, n_points)
    dx = x[1] - x[0]
    p = norm.pdf(x, mu1, s1)
    q = norm.pdf(x, mu2, s2)
    return compute_jsd(p, q, dx)

# Experiment: slide two Gaussians past each other
offsets = np.linspace(0, 10, 200)

# Wide Gaussians (overlap for small offsets)
jsd_wide = [compute_jsd_gaussians(0, 1, d, 1) for d in offsets]
w1_wide = [wasserstein_distance(
    np.random.normal(0, 1, 10000), 
    np.random.normal(d, 1, 10000)) for d in offsets]

# Narrow Gaussians (barely overlap)
jsd_narrow = [compute_jsd_gaussians(0, 0.05, d, 0.05) for d in offsets]
w1_narrow = [wasserstein_distance(
    np.random.normal(0, 0.05, 10000),
    np.random.normal(d, 0.05, 10000)) for d in offsets]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Wide Gaussians
ax = axes[0]
ax.plot(offsets, jsd_wide, 'b-', linewidth=2, label='JSD')
ax.plot(offsets, np.array(w1_wide) / max(w1_wide) * max(jsd_wide), 'r-', linewidth=2, 
        label='W₁ (scaled)')
ax.set_xlabel('Distance between means')
ax.set_ylabel('Value')
ax.set_title('Wide Gaussians (σ=1): Both metrics work')
ax.legend()
ax.grid(True, alpha=0.3)

# Narrow Gaussians
ax = axes[1]
ax.plot(offsets, jsd_narrow, 'b-', linewidth=2, label='JSD')
ax.plot(offsets, np.array(w1_narrow) / max(w1_narrow) * max(jsd_narrow), 'r-', linewidth=2,
        label='W₁ (scaled)')
ax.set_xlabel('Distance between means')
ax.set_ylabel('Value')
ax.set_title('Narrow Gaussians (σ=0.05): JSD fails!')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("🔑 Critical Difference:")
print("• JSD saturates at log(2) when distributions don't overlap → ZERO gradient for G!")
print("• Wasserstein distance grows linearly with separation → ALWAYS provides gradient")
print("• In high dimensions, distributions almost never overlap → JSD almost always fails")
print("• This is why WGAN was such a breakthrough!")

In [ ]:
# Wasserstein distance as "earth moving"

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
x = np.linspace(-5, 8, 500)

scenarios = [
    ('Small move (W₁ small)', 0, 1, 1, 1),
    ('Medium move (W₁ medium)', 0, 1, 3, 1),  
    ('Large move (W₁ large)', 0, 1, 6, 1),
]

for i, (title, mu1, s1, mu2, s2) in enumerate(scenarios):
    ax = axes[i]
    p1 = norm.pdf(x, mu1, s1)
    p2 = norm.pdf(x, mu2, s2)
    
    ax.fill_between(x, p1, alpha=0.4, color='steelblue', label='Source (P)')
    ax.fill_between(x, p2, alpha=0.4, color='coral', label='Target (Q)')
    
    # Arrow showing the "earth moving"
    ax.annotate('', xy=(mu2, 0.25), xytext=(mu1, 0.25),
                arrowprops=dict(arrowstyle='->', color='black', lw=2))
    
    w1 = wasserstein_distance(
        np.random.normal(mu1, s1, 10000),
        np.random.normal(mu2, s2, 10000))
    j = compute_jsd_gaussians(mu1, s1, mu2, s2)
    
    ax.set_title(f'{title}\nW₁={w1:.2f}, JSD={j:.3f}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Wasserstein Distance = Minimum "Work" to Transform P into Q', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("The Earth Mover's Distance measures the cheapest way to move probability mass.")
print("Unlike JSD, it's proportional to how far apart the distributions actually are.")

## 7. Putting It All Together

### Summary of Key Ideas

| Concept | Key Insight |
|---------|-------------|
| **Adversarial training** | Competition between G and D drives both to improve |
| **Optimal discriminator** | $D^*(x) = p_{data}/(p_{data} + p_g)$ — Bayes-optimal classifier |
| **GAN objective** | Minimizes Jensen-Shannon divergence between $p_{data}$ and $p_g$ |
| **Mode collapse** | G exploits reverse KL by concentrating on few modes |
| **Wasserstein distance** | Provides gradients even when distributions don't overlap |
| **WGAN** | Uses Wasserstein distance + Lipschitz constraint for stable training |

### Where GANs Fit in the Generative Model Landscape

- **GANs** → Implicit density, sharp samples, unstable training
- **VAEs** (Tutorial 13) → Approximate density (ELBO), blurry samples, stable training  
- **Normalizing Flows** (next tutorial) → Exact density, invertible, architectural constraints
- **Diffusion Models** (Tutorial 19) → Iterative denoising, best quality, slow sampling

### Connection to Information Theory

From our unifying perspective:
- Training a GAN = minimizing a **divergence** between distributions
- The discriminator = a **variational bound** on the divergence  
- At equilibrium = **zero mutual information** between real/fake label and data
- The generator = **implicit compression** of data into a deterministic mapping from noise

In [ ]:
# Final summary: comparing divergence measures

from scipy.stats import entropy

# Create distributions at various separations
separations = np.linspace(0, 6, 50)
x = np.linspace(-10, 16, 2000)
dx = x[1] - x[0]

kl_vals = []
jsd_vals = []
w1_vals = []

for d in separations:
    p = norm.pdf(x, 0, 1) + 1e-10
    q = norm.pdf(x, d, 1) + 1e-10
    
    # Normalize
    p = p / (p.sum() * dx)
    q = q / (q.sum() * dx)
    
    kl = np.sum(p * np.log(p / q)) * dx
    kl_vals.append(min(kl, 15))  # cap for visualization
    
    jsd_vals.append(compute_jsd(p, q, dx))
    
    w1_vals.append(wasserstein_distance(
        np.random.normal(0, 1, 5000),
        np.random.normal(d, 1, 5000)))

plt.figure(figsize=(12, 5))
plt.plot(separations, kl_vals, 'b-', linewidth=2, label='KL divergence')
plt.plot(separations, jsd_vals, 'g-', linewidth=2, label='JSD')
plt.plot(separations, w1_vals, 'r-', linewidth=2, label='Wasserstein-1')
plt.xlabel('Distance between distribution means', fontsize=12)
plt.ylabel('Value', fontsize=12)
plt.title('Comparing Divergence Measures: KL vs JSD vs Wasserstein', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

print("Summary:")
print("• KL divergence: unbounded, asymmetric, undefined when supports don't overlap")
print("• JSD: bounded [0, log2], symmetric, but saturates → vanishing gradients")
print("• Wasserstein: unbounded, symmetric, smooth gradients everywhere")
print("\nEach GAN variant is defined by which divergence/distance it minimizes!")